# Пример - Вариации лапласиана, гриниана и резистенсов

## Подготовка

Выполните все ячейки сверху вниз. Ревизия пакета: **63d3424**.

In [1]:
from importlib.metadata import PackageNotFoundError, distribution
import json
import subprocess
import sys

source = "https://github.com/magin67/polyform_algebra/archive/63d342437b0d5973f38befc5183d7f4ea9ecf6f0.zip"
try:
    installed = distribution("polyform_algebra")
    installed_url = json.loads(installed.read_text("direct_url.json") or "{}").get("url")
except PackageNotFoundError:
    installed = None
    installed_url = None

# A tag and a development revision can report the same version string.
if installed_url != source:
    if "polyform_algebra" in sys.modules:
        raise RuntimeError("Restart the kernel before installing the pinned package revision.")
    command = [sys.executable, "-m", "pip", "install", "-q"]
    if installed is not None and installed.version == "0.9.0":
        command += ["--force-reinstall", "--no-deps"]
    subprocess.check_call(command + [source])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


Восемь точных расчётов. Проверяем прямые изменения проводимостей, восстановление сети по вариациям гриниана и резистенсов, а также допустимость заданных резистенсов при фиксированных связях призмы.


Выполняйте фрагменты сверху вниз. Блокнот устанавливает закреплённую ревизию пакета. [polyform-algebra 63d3424](https://github.com/magin67/polyform_algebra/commit/63d342437b0d5973f38befc5183d7f4ea9ecf6f0).


## Усиление одной связи

Усиливаем ребро $ab$ единичного треугольника. Сверяем остовной коэффициент и резистенс с прямым пересчётом. На пути отдельно проверяем удаление моста.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D0%BE%D0%B4%D0%BD%D0%B0-%D0%B8%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%B0%D1%8F-%D1%81%D0%B2%D1%8F%D0%B7%D1%8C#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80---%D1%83%D1%81%D0%B8%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D1%81%D0%B2%D1%8F%D0%B7%D0%B8-%D1%82%D1%80%D0%B5%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%B0).

In [3]:
from fractions import Fraction as Q
from itertools import combinations

from sympy import Matrix, diag, eye

from polyform_algebra.core import (
    DomainError, InvalidInput, green, kron_reduce, laplacian, network,
    network_from_resistance, resistance, spanning_tree_weight,
)
from polyform_algebra.variations import (
    conductance_greenian_update, resistance_constraint_polynomials,
    vary_conductances, vary_greenian_coefficients, vary_resistances,
)


def unit_graph(labels, pairs):
    return network(tuple(labels), [(a, b, 1) for a, b in pairs])


def edge_weights(graph):
    return {(a, b): c for a, b, c in graph.edges}


def tree_weight(graph):
    return spanning_tree_weight(graph).value


def green_matrix(graph):
    return Matrix(green(graph).value.values)


def resistance_matrix(graph):
    return Matrix(resistance(graph).value.values)


triangle = unit_graph("abc", combinations("abc", 2))
edge = ("a", "b")
change = Q(1)
updated_triangle = vary_conductances(triangle, {edge: change})
green_updated, ratio = conductance_greenian_update(triangle, {edge: change})
old_r = resistance_matrix(triangle)
new_r = resistance_matrix(updated_triangle)
assert ratio == 1 + change * old_r[0, 1] == Q(5, 3)
assert green_updated == green_matrix(updated_triangle)
assert new_r[0, 1] == old_r[0, 1] / ratio == Q(2, 5)
assert tree_weight(updated_triangle) == ratio * tree_weight(triangle) == 5

path3 = unit_graph("abc", (("a", "b"), ("b", "c")))
assert len(vary_conductances(path3, {edge: -1}).components) == 2
try:
    conductance_greenian_update(path3, {edge: -1})
except DomainError:
    print("Single edge: R(a,b) =", new_r[0, 1], "; removing a bridge disconnects the path")
else:
    raise AssertionError("a disconnected output has no connected update")

Single edge: R(a,b) = 2/5 ; removing a bridge disconnects the path


## Совместное изменение связей

Усиливаем два ребра треугольника и проверяем смешанный член остовной нормировки. На пути проверяем совместное изменение, у которого один промежуточный граф распадается.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D0%B4%D0%B2%D0%B5-%D1%81%D0%B2%D1%8F%D0%B7%D0%B8-%D0%B8-%D1%81%D0%BE%D0%B2%D0%BC%D0%B5%D1%81%D1%82%D0%BD%D0%B0%D1%8F-%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D1%8F#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80---%D0%B4%D0%B2%D0%B5-%D1%83%D1%81%D0%B8%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D1%81%D0%B2%D1%8F%D0%B7%D0%B8-%D1%82%D1%80%D0%B5%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%B0).

In [4]:
# The final graph is connected even though deleting ab first breaks the path.
changes = {("a", "b"): 1, ("a", "c"): 1}
directions = Matrix([[1, 1], [-1, 0], [0, -1]])
combined = vary_conductances(triangle, changes)
updated_green, ratio = conductance_greenian_update(triangle, changes)
gram = directions.T * green_matrix(triangle) * directions
assert updated_green == green_matrix(combined)
assert ratio == 1 + gram.trace() + gram.det() == Q(8, 3)
assert tree_weight(combined) == 8 and resistance_matrix(combined)[0, 1] == Q(3, 8)

changes = {("a", "b"): -1, ("a", "c"): 2}
joint_graph = vary_conductances(path3, changes)
joint_green, factor = conductance_greenian_update(path3, changes)
gram = directions.T * green_matrix(path3) * directions
first, second = changes.values()
expected = 1 + first*gram[0, 0] + second*gram[1, 1] + first*second*gram.det()
assert joint_green == green_matrix(joint_graph)
assert factor == expected == tree_weight(joint_graph) / tree_weight(path3) == 2
print("Joint edge variation: triangle tree weight =", tree_weight(combined), "; path ratio =", factor)

Joint edge variation: triangle tree weight = 8 ; path ratio = 2


## Один коэффициент гриниана

Меняем $\gamma_{ab}$ на пути из четырёх вершин. Сверяем восстановленные проводимости и остовной коэффициент с точными значениями из заметки.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D0%BE%D0%B4%D0%B8%D0%BD-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80---%D0%BE%D0%B4%D0%B8%D0%BD-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%BF%D1%83%D1%82%D0%B8).

In [5]:
path4 = unit_graph("abcd", (("a", "b"), ("b", "c"), ("c", "d")))
delta = Q(1, 5)
result = vary_greenian_coefficients(path4, {edge: delta})
direction = Matrix([1, -1, 0, 0])
original_green = green_matrix(path4)
factor = 1 + delta * (direction.T * Matrix(laplacian(path4).value.values) * direction)[0]
assert -original_green[0, 1] == -Q(1, 8)
assert -green_matrix(result)[0, 1] == Q(3, 40)
assert edge_weights(result) == {("a", "b"): Q(2, 5), ("a", "c"): Q(1, 5),
                                ("b", "c"): Q(7, 10), ("c", "d"): 1}
assert factor == 2 and tree_weight(result) == Q(1, 2)
print("One Greenian coefficient:", -original_green[0, 1], "to", -green_matrix(result)[0, 1])

One Greenian coefficient: -1/8 to 3/40


## Два коэффициента гриниана

Увеличиваем два коэффициента гриниана треугольника на $1/3$. Сравниваем целевую матрицу с гринианом восстановленного графа и проверяем остовную нормировку.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D1%81%D0%BE%D0%B2%D0%BC%D0%B5%D1%81%D1%82%D0%BD%D0%B0%D1%8F-%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D1%8F-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D0%BE%D0%B2-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80---%D0%B4%D0%B2%D0%B0-%D0%BA%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82%D0%B0-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D1%82%D1%80%D0%B5%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA%D0%B0).

In [6]:
lap = Matrix(laplacian(triangle).value.values)
gamma = {("a", "b"): Q(1, 3), ("a", "c"): Q(1, 3)}
increments = diag(*gamma.values())
joint = vary_greenian_coefficients(triangle, gamma)
expected_green = green_matrix(triangle) + directions * increments * directions.T
factor = (eye(2) + increments * directions.T * lap * directions).det()
assert green_matrix(joint) == expected_green
assert tree_weight(joint) * factor == tree_weight(triangle)
assert factor == 8 and tree_weight(joint) == Q(3, 8)
assert edge_weights(joint) == {("a", "b"): Q(1, 4), ("a", "c"): Q(1, 4), ("b", "c"): Q(5, 8)}
print("Joint Greenian variation: determinant factor =", factor)

Joint Greenian variation: determinant factor = 8


## Один резистенс и квадросвязность

Меняем только $R_{ab}$. Проверяем, что остальные резистенсы сохранились. На исходном треугольнике сравниваем две обратные матрицы чувствительности.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D0%BE%D0%B4%D0%B8%D0%BD-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81-%D0%B8-%D0%BA%D0%B2%D0%B0%D0%B4%D1%80%D0%BE%D1%81%D0%B2%D1%8F%D0%B7%D0%BD%D0%BE%D1%81%D1%82%D1%8C#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80---%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D1%82%D1%81%D1%8F-%D1%82%D0%BE%D0%BB%D1%8C%D0%BA%D0%BE-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81-%D0%BE%D0%B4%D0%BD%D0%BE%D0%B9-%D1%81%D1%82%D0%BE%D1%80%D0%BE%D0%BD%D1%8B).

In [7]:
one = vary_resistances(triangle, {edge: Q(1, 3)})
original_r = resistance_matrix(triangle)
one_r = resistance_matrix(one)
assert one_r - original_r == Matrix([[0, Q(1, 3), 0], [Q(1, 3), 0, 0], [0, 0, 0]])
assert tree_weight(triangle) / tree_weight(one) == Q(5, 4)

# The differential matrices are indexed by the same ordered list of vertex pairs.
pairs = list(combinations(range(3), 2))
incidence = Matrix.hstack(directions, Matrix([0, 1, -1]))
transfer = (incidence.T * green_matrix(triangle) * incidence).applyfunc(lambda value: value**2)
quadric = Matrix([[(lap[i, k]*lap[j, l] + lap[i, l]*lap[j, k])/2 for k, l in pairs]
                  for i, j in pairs])
assert transfer == Matrix([[4, 1, 1], [1, 4, 1], [1, 1, 4]]) / 9
assert transfer * quadric == eye(3)
print("One resistance: R(a,b) =", one_r[0, 1], "; quadric matrices are inverse")

One resistance: R(a,b) = 1 ; quadric matrices are inverse


## Два резистенса

Сверяем совместное и последовательное восстановление. При большем изменении проверяем допустимость совместного результата и вырождение обоих промежуточных состояний.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D1%81%D0%BE%D0%B2%D0%BC%D0%B5%D1%81%D1%82%D0%BD%D0%BE%D0%B5-%D0%B8%D0%B7%D0%BC%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D1%81-%D0%B4%D0%B2%D1%83%D0%BC%D1%8F-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%B0%D0%BC%D0%B8).

In [8]:
deltas = {("a", "b"): Q(1, 3), ("a", "c"): Q(1, 3)}
result = vary_resistances(triangle, deltas)
sequential = vary_resistances(one, {("a", "c"): Q(1, 3)})
assert edge_weights(result) == edge_weights(sequential) == {
    ("a", "b"): Q(3, 5), ("a", "c"): Q(3, 5), ("b", "c"): Q(6, 5),
}
assert tree_weight(result) == Q(9, 5)
valid_joint = vary_resistances(triangle, {("a", "b"): 2, ("a", "c"): 2})
assert len(valid_joint.components) == 1
for pair in deltas:
    try:
        vary_resistances(triangle, {pair: 2})
    except InvalidInput:
        pass
    else:
        raise AssertionError("the intermediate metric is singular")
print("Joint resistance variation: tree weight =", tree_weight(result))

Joint resistance variation: tree weight = 9/5


## Допустимое изменение призмы

Сохраняем связи граничного треугольника и запрещённые пары. Проверяем точные резистенсы и сравниваем их с результатом исключения внутренних вершин.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BF%D1%80%D0%B8-%D0%BE%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%B5%D0%BD%D0%B8%D1%8F%D1%85-%D0%BD%D0%B0-%D1%81%D0%B2%D1%8F%D0%B7%D0%B8#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BF%D1%80%D0%B8%D0%B7%D0%BC%D1%8B).

In [9]:
boundary = tuple("abc")
fixed = tuple(combinations(boundary, 2))
variable = (("d", "e"), ("d", "f"), ("e", "f"), ("a", "d"), ("b", "e"), ("c", "f"))
prism = unit_graph("abcdef", (*fixed, *variable))


def target(k):
    r = Q(8, 15)
    return {("a", "b"): k*r, ("a", "c"): r, ("b", "c"): r}


def witness(k, g):
    a = (4*k*k - 31*k + 30) / (4*k*(4-k))
    b = (4*k - 1) / (4*(4-k))
    nu = g*b / (g-3*b)
    x = g*(a*g - b*(2*a+b)) / ((g-2*a-b)*(g-3*b))
    return (x, nu, nu, g, g, g)


def apply_witness(weights):
    return vary_conductances(prism, {pair: value-1 for pair, value in zip(variable, weights, strict=True)})


k = Q(9, 8)
values = witness(k, Q(7))
altered = apply_witness(values)
actual = edge_weights(altered)
assert values == (0, Q(7, 20), Q(7, 20), 7, 7, 7)
assert all(actual[pair] == 1 for pair in fixed)
assert set(actual) <= set(fixed + variable) and all(value >= 0 for value in actual.values())
assert resistance(altered, pairs=tuple(fixed)).value.values == tuple(target(k).values())
reduced = kron_reduce(altered, boundary).value.network
assert resistance(reduced, pairs=tuple(fixed)).value.values == tuple(target(k).values())
assert tree_weight(altered) == Q(7203, 4)
print("Constrained prism: c(de) =", values[0], "; tree weight =", tree_weight(altered))

Constrained prism: c(de) =

 0 ; tree weight = 7203/4


## Точные сертификаты ограничений

Устраняем знаменатели с помощью остовного полинома и проверяем два точных решения. Для $k=2$ требуемая редуцированная проводимость $ab$ равна нулю, хотя прямая связь уже имеет проводимость один.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D0%BF%D1%80%D0%B5%D0%BE%D0%B1%D1%80%D0%B0%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D1%8F-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2/%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B8-%D0%BB%D0%B0%D0%BF%D0%BB%D0%B0%D1%81%D0%B8%D0%B0%D0%BD%D0%B0,-%D0%B3%D1%80%D0%B8%D0%BD%D0%B8%D0%B0%D0%BD%D0%B0-%D0%B8-%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D1%81%D0%BE%D0%B2/%D0%B0%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC-%D0%B2%D0%B0%D1%80%D0%B8%D0%B0%D1%86%D0%B8%D0%B9-%D0%BF%D1%80%D0%B8-%D0%BE%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%87%D0%B5%D0%BD%D0%B8%D1%8F%D1%85-%D0%BD%D0%B0-%D1%81%D0%B2%D1%8F%D0%B7%D0%B8#%D0%B2%D0%BE%D0%B7%D0%B2%D1%80%D0%B0%D1%89%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BA-%D0%BF%D1%80%D0%B8%D0%B7%D0%BC%D0%B5-%D0%B8-%D0%BF%D1%80%D0%BE%D0%B2%D0%B5%D1%80%D0%BA%D0%B0).

In [10]:
variables, tree, residuals = resistance_constraint_polynomials(prism, variable, target(k))
for stretch, g in ((Q(9, 8), Q(7)), (Q(21, 20), Q(2))):
    active = witness(stretch, g)
    exact_graph = apply_witness(active)
    assignment = dict(zip(variables, active, strict=True))
    tree_at_witness = tree.subs(assignment)
    assert min(active) >= 0 and tree_at_witness == tree_weight(exact_graph) > 0
    # Residuals above use target(k). Substitute the new target without
    # rebuilding the three separating-forest numerators.
    for original, new, residual in zip(target(k).values(), target(stretch).values(), residuals, strict=True):
        assert residual.subs(assignment) + (original-new)*tree_at_witness == 0

# k = 2 would require zero reduced conductance ab, below its fixed direct value.
stretch = Q(2)
r_ab, r_ac, r_bc = target(stretch).values()
target_matrix = [[0, r_ab, r_ac], [r_ab, 0, r_bc], [r_ac, r_bc, 0]]
target_graph = network_from_resistance(target_matrix, boundary).value
assert edge_weights(target_graph).get(("a", "b"), 0) == 0
assert edge_weights(prism)[("a", "b")] == 1
print("Polynomial constraints: two exact witnesses; k = 2 violates the fixed edge bound")

Polynomial constraints: two exact witnesses; k = 2 violates the fixed edge bound
